# Setup

In [ ]:
# Tìm thư mục gốc của dự án để notebook biết nơi chứa mã nguồn và dữ liệu.
from pathlib import Path
import sys
import cv2
import pandas as pd
import numpy as np
from tqdm import tqdm

def find_project_root(max_up=6):
    p = Path.cwd()
    for _ in range(max_up + 1):
        if (p / "src").exists() and (p / "configs").exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    return Path.cwd()

PROJECT_ROOT = find_project_root()

sys.path.append(str(PROJECT_ROOT))

from src.config import load_yaml
from src.io_utils import save_csv

pipeline_cfg = load_yaml(PROJECT_ROOT / "configs/pipeline_config.yaml")
tracklet_cfg = pipeline_cfg["tracklet"]

tracks_df = pd.read_csv(PROJECT_ROOT / "outputs/tracks/tracks.csv")

tracks_df.head()


# Tạo tracklet summary

In [ ]:
# Gom các detections liên tiếp của cùng một track thành tracklet và đánh giá chất lượng.
tracklet_rows = []

for (camera_id, track_id), group in tracks_df.groupby(["camera_id", "local_track_id"]):
    group = group.sort_values("frame_id")

    num_frames = len(group)
    start_frame = int(group["frame_id"].min())
    end_frame = int(group["frame_id"].max())
    start_time = float(group["timestamp_sec"].min())
    end_time = float(group["timestamp_sec"].max())
    duration = end_time - start_time

    avg_conf = float(group["confidence"].mean())
    avg_height = float((group["y2"] - group["y1"]).mean())
    avg_area = float(((group["x2"] - group["x1"]) * (group["y2"] - group["y1"])).mean())

    valid = (
        num_frames >= tracklet_cfg["min_frames"] and
        duration >= tracklet_cfg["min_duration_sec"] and
        avg_height >= tracklet_cfg["min_bbox_height"]
    )

    quality_score = min(1.0, avg_conf * 0.7 + min(avg_height / 200.0, 1.0) * 0.3)

    tracklet_rows.append({
        "camera_id": camera_id,
        "local_track_id": int(track_id),
        "start_frame": start_frame,
        "end_frame": end_frame,
        "start_time": start_time,
        "end_time": end_time,
        "duration_sec": duration,
        "num_frames": num_frames,
        "avg_confidence": avg_conf,
        "avg_bbox_height": avg_height,
        "avg_bbox_area": avg_area,
        "quality_score": quality_score,
        "valid": bool(valid)
    })

tracklets_df = pd.DataFrame(tracklet_rows)

out_path = PROJECT_ROOT / "outputs/tracklets/tracklets_summary.csv"
save_csv(tracklets_df, out_path)

print("Saved:", out_path)
tracklets_df.head()


# Hàm crop quality

In [ ]:
# Định nghĩa hàm kiểm tra chất lượng crop và cắt vùng ảnh an toàn từ bounding box.
def compute_sharpness(image):
    if image is None or image.size == 0:
        return 0.0

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()


def safe_crop(frame, x1, y1, x2, y2):
    h, w = frame.shape[:2]

    x1 = max(0, min(int(x1), w - 1))
    x2 = max(0, min(int(x2), w - 1))
    y1 = max(0, min(int(y1), h - 1))
    y2 = max(0, min(int(y2), h - 1))

    if x2 <= x1 or y2 <= y1:
        return None

    return frame[y1:y2, x1:x2]


def crop_quality(row, frame):
    crop = safe_crop(frame, row.x1, row.y1, row.x2, row.y2)

    if crop is None:
        return 0.0, None

    h, w = frame.shape[:2]
    bbox_area = max(0, row.x2 - row.x1) * max(0, row.y2 - row.y1)
    norm_area = bbox_area / (w * h)

    sharpness = compute_sharpness(crop)

    score = float(row.confidence) + 2.0 * norm_area + 0.0001 * sharpness

    return float(score), crop


# Chọn top-K representative crops

In [ ]:
# Chọn crop chất lượng cao nhất cho mỗi tracklet, dùng để trích embedding và đối chiếu.
crop_root = PROJECT_ROOT / "outputs/crops"
crop_root.mkdir(parents=True, exist_ok=True)

top_k = tracklet_cfg["top_k_crops"]
representative_rows = []

valid_tracklets = tracklets_df[tracklets_df["valid"] == True]

for _, trow in tqdm(valid_tracklets.iterrows(), total=len(valid_tracklets)):
    camera_id = trow["camera_id"]
    track_id = int(trow["local_track_id"])

    group = tracks_df[
        (tracks_df["camera_id"] == camera_id) &
        (tracks_df["local_track_id"] == track_id)
    ].copy()

    candidates = []

    for _, row in group.iterrows():
        frame = cv2.imread(row["image_path"])

        if frame is None:
            continue

        score, crop = crop_quality(row, frame)

        if crop is None or crop.size == 0:
            continue

        candidates.append((score, row, crop))

    candidates = sorted(candidates, key=lambda x: x[0], reverse=True)[:top_k]

    save_dir = crop_root / camera_id / f"track_{track_id}"
    save_dir.mkdir(parents=True, exist_ok=True)

    for rank, (score, row, crop) in enumerate(candidates):
        crop_path = save_dir / f"rank_{rank + 1}_frame_{int(row.frame_id)}.jpg"
        cv2.imwrite(str(crop_path), crop)

        representative_rows.append({
            "camera_id": camera_id,
            "local_track_id": track_id,
            "frame_id": int(row.frame_id),
            "timestamp_sec": float(row.timestamp_sec),
            "crop_path": str(crop_path),
            "quality_score": float(score),
            "confidence": float(row.confidence)
        })

rep_df = pd.DataFrame(representative_rows)

out_path = PROJECT_ROOT / "outputs/crops/representative_crops.csv"
save_csv(rep_df, out_path)

print("Saved:", out_path)
rep_df.head()


# Some check

In [ ]:
# Cell này thực hiện một bước cụ thể trong luồng pipeline, có thể là nạp dữ liệu, xử lý, hoặc lưu kết quả.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

crop_dirs = [
    PROJECT_ROOT / "outputs" / "crops",
    PROJECT_ROOT / "outputs" / "person_crops",
]

for crop_root in crop_dirs:
    if crop_root.exists():
        imgs = list(crop_root.rglob("*.jpg")) + list(crop_root.rglob("*.png"))
        print("Crop root:", crop_root)
        print("Number of crop images:", len(imgs))
        print("First 10 crops:")
        for p in imgs[:10]:
            print(p)
